### Handwritten Digit Classification using CNN & RNN

In [56]:
import pandas as pd 
import numpy as np 
import torch 
import torch.nn as nn 
import torch.optim as optim

### Loading the dataset

In [57]:
import torchvision
from torchvision import datasets,transforms
from torchvision.datasets import MNIST


In [58]:
transform=transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.1307),(0.3081))
    ]
)

In [59]:
trainset=MNIST(root="./data",transform=transform, train=True,download=True)
testset=MNIST(root="./data",transform=transform,train=False,download=True)

### Datasets and Dataloaders 

In [60]:
from torch.utils.data import DataLoader

In [61]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.1307, std=0.3081)
           )

In [62]:
testset

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.1307, std=0.3081)
           )

In [63]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

### Defining RNN Model

In [64]:
class RNN(nn.Module):
    def __init__(self,input_size=28,hidden_size=128,num_layers=1,num_classes=10):
        super().__init__()


        self.hidden_size=hidden_size
        self.num_layers=num_layers


        # RNN layer

        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        # Fully connected layer

        self.fc=nn.Linear(hidden_size,num_classes)
    
    def forward(self,x):
        # x.shape => (batch_size,1,28,28)

        x=x.squeeze(1)
        
        h0=torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size
        )

        # RNN Forward

        out,hidden=self.rnn(x,h0)

        out=out[:,-1,:]

        out=self.fc(out)

        return out



In [65]:
model=RNN()

In [66]:
print(model)

RNN(
  (rnn): RNN(28, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)


In [70]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

### Training the Model 

In [71]:
epochs = 10 

for epoch in range(epochs):
    model.train()

    training_loss=0.0

    for images,labels in trainloader:

        optimizer.zero_grad()

        outputs=model(images)

        loss=criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        training_loss+=loss.item()


    print(f"{epoch+1}/{epochs},loss={training_loss/len(trainloader)}")





1/10,loss=0.6862634533662786
2/10,loss=0.29349863491475836
3/10,loss=0.2241416239360375
4/10,loss=0.18917759601026773
5/10,loss=0.16623872154747754
6/10,loss=0.15188870028650234
7/10,loss=0.1395584496705215
8/10,loss=0.13251397865357747
9/10,loss=0.12586273234595718
10/10,loss=0.11454830353440189


In [73]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in testloader:

        outputs = model(images)

        # Predicted class
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 96.82%
